In [3]:
import pandas as pd
import glob
import os
import functools as ft
from typing import List
import re

def process_mitosis_csv(csv_file):
    df = pd.read_csv(csv_file)

    # 20211114_e14WTCelsr488Fz555Vang647_1_1_0001_FF_cp_masks.tif

    filename_parts = df['filename'].str.split('_', expand=True)

    # Extract metadata
    df['date'] = filename_parts[0]
    df['age'] = filename_parts[1].str.slice(start=0, stop=3)
    df['variant'] = filename_parts[1].str.slice(start=3,stop=5)
    pattern = r'EEA1647_(.+?)_cp_masks\.tif'
    df['image_number'] = df['filename'].str.extract(pattern, expand=False)
    df = df.rename(columns={'label_integer' : 'roi'}).reset_index()

    return df

def process_colocalization_data(
    df_celsr1: pd.DataFrame,
    df_Fz6: pd.DataFrame,
    df_EEA1: pd.DataFrame
) -> pd.DataFrame:
    """
    Process colocalization data from multiple DataFrames and compute counts.
    
    Parameters
    ----------
    df_celsr1 : pd.DataFrame
        DataFrame containing Celsr1 data with multiple masked channels
    df_Fz6 : pd.DataFrame
        DataFrame containing Fz6 data
    df_EEA1 : pd.DataFrame
        DataFrame containing EEA1 data
    
    Returns
    -------
    pd.DataFrame
        Merged DataFrame with counts grouped by age, variant, date, and roi
    """
    
    # Define column configurations
    column_configs = {
        'celsr1_base': ['LabelObj', 'original_filename', 'date', 'age', 
                        'variant', 'roi', 'image_number'],
        'intensity_patterns': {
            'Celsr1': ['Max', 'Avg', 'Sum'],
            'Fz6': ['Max', 'Avg', 'Sum'],
            'EEA1': ['Max', 'Avg', 'Sum']
        }
    }
    
    # Helper function to generate intensity column names
    def get_intensity_cols(source: str, target: str, patterns: List[str]) -> List[str]:
        return [f'Intensity{p}{source}-{target}' for p in patterns]
    
    # Process Celsr1 channels
    celsr1_channels = {
        1: ('Celsr1', 'Celsr1'),  # CC
        2: ('Celsr1', 'Fz6'),    # CF
        3: ('Celsr1', 'EEA1')     # CV
    }
    
    celsr1_subsets = []
    for mask_channel, (source, target) in celsr1_channels.items():
        
        df_filtered = df_celsr1[df_celsr1['masked_channel'].astype(int) == mask_channel]

        # Build column list
        cols = ['LabelObj', 'original_filename']
        cols.extend(get_intensity_cols(source, target, column_configs['intensity_patterns'][target]))
        
        # Add base columns only for the first channel (CC)
        if mask_channel == 1:
            cols.extend(['date', 'age', 'variant', 'roi', 'image_number'])
        
        celsr1_subsets.append(df_filtered[cols])
    
    # Merge Celsr1 subsets
    df_celsr1_merged = ft.reduce(
        lambda left, right: pd.merge(left, right, on=['LabelObj', 'original_filename'], how='outer'),
        celsr1_subsets
    )
    
    # Process Fz6 and Vangl2 DataFrames
    def process_single_protein(df: pd.DataFrame, protein: str) -> pd.DataFrame:
        cols = ['LabelObj', 'original_filename']
        cols.extend(get_intensity_cols(protein, protein, column_configs['intensity_patterns'][protein]))
        cols.extend(['date', 'age', 'variant', 'roi', 'image_number'])
        return df[cols]
    
    df_Fz6_subset = process_single_protein(df_Fz6, 'Fz6')
    df_EEA1_subset = process_single_protein(df_EEA1, 'EEA1')
    
    # Calculate counts for each dataset
    groupby_cols = ['age', 'variant', 'date', 'roi', 'image_number']
    
    # Celsr1 counts (multiple intensity columns)
    celsr1_intensity_cols = [
        'IntensityMaxCelsr1-Celsr1',
        'IntensityMaxCelsr1-Fz6',
        'IntensityMaxCelsr1-EEA1'
    ]
    
    df_celsr1_count = df_celsr1_merged.groupby(groupby_cols)[celsr1_intensity_cols].apply(
        lambda x: (x > 0).sum()
    ).rename(columns={col: f'N_{col.replace("IntensityMax", "")}' for col in celsr1_intensity_cols})
    
    # Fz6 counts
    df_Fz6_count = df_Fz6_subset.groupby(groupby_cols)['IntensityMaxFz6-Fz6'].apply(
        lambda x: (x > 0).sum()
    ).to_frame().rename(columns={'IntensityMaxFz6-Fz6': 'N_Fz6-Fz6'})
    
    # Vangl2 counts
    df_EEA1_count = df_EEA1_subset.groupby(groupby_cols)['IntensityMaxEEA1-EEA1'].apply(
        lambda x: (x > 0).sum()
    ).to_frame().rename(columns={'IntensityMaxEEA1-EEA1': 'N_EEA1-EEA1'})
    
    # Merge all count DataFrames
    count_dfs = [df_celsr1_count, df_Fz6_count, df_EEA1_count]
    df_merged = ft.reduce(
        lambda left, right: pd.merge(left, right, on=groupby_cols, how='left'),
        count_dfs
    ).reset_index().fillna(0)
    
    return df_merged



def ensure_dataframe(obj):
    """
    Converts a Pandas Series to a DataFrame if it is a Series, 
    otherwise returns the object unchanged.
    """
    if isinstance(obj, pd.Series):
        return obj.to_frame()
    else:
        return obj
    
def process_csv_files(directory_path, output_path):
    """
    Process CSV files with MASK_C prefix and merge data based on object_channel
    
    Parameters:
    directory_path (str): Path to directory containing CSV files
    
    Returns:
    dict: Dictionary with keys 'Celsr1', 'Fz6', 'EEA1' containing respective dataframes
    """
    
    # Channel mapping
    channel_map = {1: 'Celsr1', 2: 'Fz6', 3: 'EEA1'}
    
    # Find all CSV files with MASK_C prefix
    csv_pattern = os.path.join(directory_path, "MASK_C*.csv")
    csv_files = glob.glob(csv_pattern)
    
    if not csv_files:
        print(f"No CSV files found with pattern MASK_C*.csv in {directory_path}")
        return {}
    
    # Dictionary to store data for each object channel
    channel_data = {'Celsr1': [], 'Fz6': [], 'EEA1': []}
    
    groupbys = {'Celsr1': ['IntensityMaxCelsr1-Celsr1', 'IntensityMaxCelsr1-Fz6', 'IntensityMaxCelsr1-EEA1'],
                'Fz6'  : 'IntensityMaxFz6-Fz6',
                'EEA1'  : 'IntensityMaxEEA1-EEA1',
                }
    
    desired_cols = [item for sublist in groupbys.values() 
          for item in (sublist if isinstance(sublist, list) else [sublist])]
    
    # Process each CSV file
    for file_path in csv_files:
        filename = os.path.basename(file_path)
        
        try:
            
            object_channel = filename.split("-")[0][-1]
            masked_channel = filename.split("_Masked_C")[1][0]

            # Create original_filename (remove first 8 characters, split by "_ObjectProbabilities")
            original_filename = filename[8:].split("_ObjectProbabilities")[0]
            
            # Parse filename components
            filename_parts = original_filename.split('_')
            
            if len(filename_parts) < 4:
                print(f"Filename does not have expected format: {filename}")
                continue
            
            # Extract metadata
            date = filename_parts[0]
            
            age = filename_parts[1][:3]
            variant = filename_parts[1][3:5]
            roi = int(filename_parts[-1])

            
            pattern = r'EEA1647_(.+?)_object'
            #df['image_number'] = df['filename'].str.extract(pattern, expand=False)
            #image_number = '_'.join(filename_parts[2:5])
            image_number = re.search(pattern, original_filename).group(1)
            # Load CSV file 
            df = pd.read_csv(file_path, usecols=[2,4,5,7])
            
            # Add metadata columns
            df['original_filename'] = original_filename
            df['date'] = date
            df['age'] = age
            df['variant'] = variant
            df['roi'] = roi
            df['image_number'] = image_number
            df['object_channel'] = object_channel
            df['masked_channel'] = masked_channel
            df['source_file'] = filename
            

            channel_data[channel_map[int(object_channel)]].append(df)
            
        except Exception as e:
            print(f"Error processing file {filename}: {str(e)}")
            continue
    
    # Create final dataframes for each channel
    final_dataframes = {}
    
    for channel in ['Celsr1', 'Fz6', 'EEA1']:
        if not channel_data[channel]:
            print(f"No data found for channel {channel}")
            final_dataframes[channel] = pd.DataFrame()
            continue
        
        # Combine all dataframes for this channel
        channel_df = pd.concat(channel_data[channel], ignore_index=True)
        channel_df.to_csv(f"{output_path}/{channel}_merged.csv")
        final_dataframes[channel] = channel_df
    
    return process_colocalization_data(final_dataframes['Celsr1'], final_dataframes['Fz6'], final_dataframes['EEA1'])

# Example usage
def main():
    # Specify the directory containing your CSV files
    #directory_path = r"C:/Users/wgiang/Documents/09112025-Will/3D-coloc"  
    directory_path = r"D:/Sarah/EEA1/coloc"  
    # Specify the path to your cell cycle CSV file
    #mitosis_csv = r"C:/Users/wgiang/Documents/09112025-Will/CFVMitoticCellsCSV.csv"
    mitosis_csv = r"D:/Sarah/EEA1/10282025_EEA1_MitoticCellsCSV.csv"
    # Specify the directory where you want your output files to be saved
    #output_path = r"C:/Users/wgiang/Documents/09112025-Will/2025-09-11_output" 
    output_path = r"D:/Sarah/EEA1/coloc-data-output" 


    # Process the files
    dataframe = process_csv_files(directory_path, output_path)
    dataframe.to_csv(output_path + "/dataframe-before-cellcycleaddition.csv")

    
    df_mitosis = process_mitosis_csv(mitosis_csv)
    df_mitosis.to_csv(output_path+"/mitosis_info.csv")

    df_final = pd.merge(
        dataframe, df_mitosis[['date','age','variant','roi','image_number','mitotic_phase']], how='left',on=['date', 'age', 'variant', 'roi', 'image_number'])
    
    # Move the column name to earlier
    col = df_final.pop("mitotic_phase")
    df_final.insert(5, col.name, col)
  
    df_final.to_csv(output_path+r"/merged_counts_with_cell-cycle.csv",index=False)

if __name__ == "__main__":
    main()
